# EDA on Olympic data


## Create Data Frame
- Read as CSV using Spark
- Display using display()

In [0]:
%python
DATA_PATH = '/Volumes/data/olympic_games/raw_data'
ATHLETE_PATH = DATA_PATH + '/athlete_events.csv'

df_olympics = spark.read.csv(ATHLETE_PATH, header=True)

df_olympics.display()

## Schema for data frame
Since all is String we import a schema to determin columntype
- Check in frame above to se what it should be.
- Use StructType to create shema
- Create new df with schema

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, ShortType, ByteType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_olympics_schema = spark.read.csv(ATHLETE_PATH, header=True, schema=schema)

df_olympics_schema.display()

## Control null-vales
- Check all null-values and sum them up

#### Functions explained
- df_olympics_schema.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_olympics_schema.columns])

| Function | Does | 
|----------|------| 
|col()|Makes it possible to run funktions|
|isNull()|Returns True if Null else False|
|cast("int")|Translate True to 1 False to 0|
|sum()|Sum the value of every column|
|alias()|Change header-title|


In [0]:
%python
from pyspark.sql.functions import col, sum

nulls = df_olympics_schema.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_olympics_schema.columns])

display(nulls)

### Swedish Olympians

In [0]:
df_olympic_sweds = df_olympics_schema.filter("NOC = 'SWE'")

df_olympic_sweds.display()

### Most olympians in the north

In [0]:
df_olympic_most_north = df_olympics_schema.groupBy("NOC").count().filter("NOC IN ('SWE','NOR','FIN','DEN','ISL')")
                                                                       
df_olympic_most_north.display()                                                                    

### Swedish Olympian medalists

In [0]:
df_olympics_schema.createOrReplaceTempView("df_olympics_schema")

df_swe_medalist = spark.sql("""
                          SELECT
                            sport,
                            Name,
                            medal
                          FROM df_olympics_schema
                          WHERE
                            NOC = 'SWE' AND medal IN ('Gold','Silver','Bronze')
                          """)
df_swe_medalist.display()

### Numbers of medals to Sweden

In [0]:
df_swe_medals = spark.sql("""
                          SELECT
                            sport,
                            count(medal) as medals
                          FROM df_olympics_schema
                          WHERE
                            NOC = 'SWE' AND medal IN ('Gold','Silver','Bronze')
                          GROUP BY
                            sport
                          ORDER BY
                            medals DESC
                          """)
df_swe_medals.display()

In [0]:
fig = df_swe_medals.plot(kind="bar", y="sport", x="medals", title="Swedish Medals")

fig.update_layout(yaxis= {'autorange': 'reversed'})
fig.show()

## With SQL

In [0]:
%sql
FROM df_olympics_schema LIMIT 10;

In [0]:
%sql

SHOW CATALOGS;